# Mandatory Assignment 2: Ensemble Models & Artificial Neural Networks
#### Group: 04
Group members: Brage Blomkvist, Patrick Rafael Jemieljanczyk & William Bjørling Nilsen

In [13]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier


from sklearn.tree import DecisionTreeClassifier


## Description of Dataset

The dataset used is a set of daily weather observations from Australia. The observations are taken from a span of ten years. Various different locations are used

The dataset can be found here: https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package/data

### Problem

In [14]:
# Read the dataset
dataset_AUS = pd.read_csv('Dataset/weatherAUS.csv')
# Uses existing column names baked into csv file.

# Show the first 5 rows to validate that the data import has been successful
dataset_AUS.head()



,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


## Data Processing

In [15]:
# Overview of dataset
# Look at the implementation details, data types and missing data
dataset_AUS.info()

<class 'pandas.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  str    
 1   Location       145460 non-null  str    
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  str    
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  str    
 10  WindDir3pm     141232 non-null  str    
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null   float64


In [16]:
# See how many different locations are used.
dataset_AUS.value_counts("Location").nunique()


10

In [17]:
# Check if every location has the same amount of observations
dataset_AUS.value_counts("Location")

Location
Canberra            3436
Sydney              3344
Melbourne           3193
Brisbane            3193
Adelaide            3193
Perth               3193
Hobart              3193
Darwin              3193
Albury              3040
Wollongong          3040
MountGinini         3040
Ballarat            3040
Bendigo             3040
Cairns              3040
GoldCoast           3040
Townsville          3040
MountGambier        3040
Albany              3040
Launceston          3040
AliceSprings        3040
Newcastle           3039
Penrith             3039
Tuggeranong         3039
BadgerysCreek       3009
Cobar               3009
CoffsHarbour        3009
Moree               3009
NorfolkIsland       3009
Richmond            3009
SydneyAirport       3009
WaggaWagga          3009
Williamtown         3009
Sale                3009
MelbourneAirport    3009
Mildura             3009
Portland            3009
Watsonia            3009
Dartmoor            3009
Nuriootpa           3009
Woomera         

### Missing Data

In [18]:
# Look at the amount of missing values
dataset_AUS.isna().sum()

Date                 0
Location             0
MinTemp           1485
MaxTemp           1261
Rainfall          3261
Evaporation      62790
Sunshine         69835
WindGustDir      10326
WindGustSpeed    10263
WindDir9am       10566
WindDir3pm        4228
WindSpeed9am      1767
WindSpeed3pm      3062
Humidity9am       2654
Humidity3pm       4507
Pressure9am      15065
Pressure3pm      15028
Cloud9am         55888
Cloud3pm         59358
Temp9am           1767
Temp3pm           3609
RainToday         3261
RainTomorrow      3267
dtype: int64

In [19]:
dataset_AUS.count()

Date             145460
Location         145460
MinTemp          143975
MaxTemp          144199
Rainfall         142199
Evaporation       82670
Sunshine          75625
WindGustDir      135134
WindGustSpeed    135197
WindDir9am       134894
WindDir3pm       141232
WindSpeed9am     143693
WindSpeed3pm     142398
Humidity9am      142806
Humidity3pm      140953
Pressure9am      130395
Pressure3pm      130432
Cloud9am          89572
Cloud3pm          86102
Temp9am          143693
Temp3pm          141851
RainToday        142199
RainTomorrow     142193
dtype: int64

### Splitting the Dataset
Created temporary splitting of data


In [20]:
# Gridsearch can be used, but Nested Cross validation should be considered first. Must use Cross validation for evaluation instead of random states.

# Creating a temporary dataset
dataset_AUS_temp = pd.read_csv('Dataset/weatherAUS.csv')

# Temporary separating into targets and features
y = dataset_AUS_temp['RainTomorrow']
x = dataset_AUS_temp.drop(columns=['RainTomorrow'])

# Temporary splitting
# Split into training and testing
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
)

# Select numerical columns
numerical_features = x.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# Select categorical columns
categorical_features = x.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

# Pipelines
to do:
   - Make the processor able to remove certain columns. the one who runs the program can then choose between removing the columns or not, to see the differences this creates in the models.


In [21]:
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder())
])

# Creating seperate preprocessor
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('categorical', categorical_pipeline, categorical_features)
])

# Pipeline without seperate processor:
'''
dt_pipe = Pipeline([
    ('preprocessing', ColumnTransformer([
        ('numerical', numerical_pipeline, numerical_features),
        ('categorical', categorical_pipeline, categorical_features)
    ])),
    ('estimator', DecisionTreeClassifier())
    ])
'''

# Decision tree pipeline
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', LogisticRegression(max_iter=500))
])

# K-nearest neighbours
knn_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', KNeighborsClassifier(
        n_neighbors=5
    ))
])

# train decision tree
#dt_pipe.fit(x_train, y_train)

### Normalization

### Encoding

## Baseline Models

In [22]:
# (Automated tuning)

## Advanced Models

## Advanced Neural Networks

## Evaluation & Comparison

### Testing

#### Combined Results Table

### Confusion Matrices & Additional Plots

### Conclusion